# Osoba C: Analiza własności modeli językowych (Część II)

Zakres (4 pkt): II.A–C i pkt 1–2 z [instrukcji](../Zadanie_2_zmodyfikowane.md). Ta część **nie zależy od Części I** (używamy modeli bazowych, bez strojenia).

## Do zrobienia

1. **II.A–C: zapoznanie się z materiałami**: zbiór `clarin-knext/wsd_polish_datasets`, artykuł o kontekstowych reprezentacjach (blog Stanford + Ethayarajh, D19-1006), Logit Lens (LessWrong + arXiv 2303.08112). Krótka notatka w notebooku.
2. **Pkt 1: odtworzyć badanie z Ethayarajha dla BERT i GPT-2** na zbiorze WSD (2 pkt):
   - a. **anizotropia**: średnie podobieństwo kosinusowe losowych par reprezentacji słów, osobno dla każdej warstwy,
   - b. **zależność od kontekstu (context-specificity)**: self-similarity (to samo słowo w różnych kontekstach), intra-sentence similarity i (jeśli starczy czasu) maksymalna wyjaśniona wariancja. Wyniki jako wykresy per warstwa, dla obu modeli.
3. **Pkt 2: Logit Lens dla GPT-2** (1,5 pkt):
   - a. model `sdadas/polish-gpt2-medium`,
   - b. **co najmniej 20 promptów**: zdania faktograficzne (np. „Stolicą Francji jest”) i sprawdzające gramatykę (np. zgodność form fleksyjnych),
   - c. dla każdej warstwy: stan ukryty ostatniego tokenu → końcowy LayerNorm (`ln_f`) → macierz wyjściowa (`lm_head`) → rozkład po słowniku,
   - d. **heatmapa** najbardziej prawdopodobnych tokenów w kolejnych warstwach dla wybranych promptów,
   - e. **ranga i prawdopodobieństwo** tokenu przewidywanego na wyjściu modelu, per warstwa, uśrednione po promptach,
   - f. wnioski: od której warstwy model zna odpowiedź i czy różni się to dla zdań faktograficznych i gramatycznych (porównaj te dwie grupy osobno).

## Ustalone

- Modele: `allegro/herbert-base-cased` (BERT) i `sdadas/polish-gpt2-medium` (GPT-2), oba w wersji bazowej, bez adapterów.
- Dane do pkt 1: `clarin-knext/wsd_polish_datasets` (biblioteka `datasets`, już w zależnościach). Reprezentacja słowa = średnia z jego subtokenów, każda warstwa (`output_hidden_states=True`).
- Wyniki pkt 1 porównujemy per warstwa, osobno dla BERT i GPT-2.
- Logit Lens: uwaga na to, że w instrukcji nazwa modelu ma literówkę (`polishgpt2-medium`), poprawna nazwa to `sdadas/polish-gpt2-medium`.
- Wykresy: Plotly (już w zależnościach) lub matplotlib/seaborn.
- Prompty do Logit Lens trzymamy w notebooku jako listę z polem `typ` (`fakt` / `gramatyka`) i oczekiwanym tokenem, żeby uśredniać osobno per typ.

## Części wspólne (`../utils/`)

Brak: ta część używa własnych danych (WSD) i modeli bazowych. Jeśli powstanie coś przydatnego dla A/B (np. wyciąganie stanów ukrytych per warstwa), dodaj to jako mały moduł w `../utils/`.


In [ ]:
import torch
import numpy as np
import pandas as pd
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModel, AutoModelForCausalLM
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
from collections import defaultdict
import random
import re

# Ustawienie ziarna losowości dla powtarzalności wyników
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Używane urządzenie: {DEVICE}")

BERT = "allegro/herbert-base-cased"
GPT2 = "sdadas/polish-gpt2-medium"

Używane urządzenie: cpu


## II.A–C: notatka z materiałów

### II.A–C: Notatka z materiałów

#### 1. Zbiór danych: `clarin-knext/wsd_polish_datasets`
Polskojęzyczny zbiór przeznaczony do zadania ujednoznaczniania znaczeń słów (*Word Sense Disambiguation* – WSD). Zawiera zdania, w których te same leksykalnie słowa występują w zróżnicowanych kontekstach i znaczeniach. Stanowi bazę do empirycznego badania, jak wielowarstwowe modele reprezentują to samo słowo w odmiennych zdaniach (podejście analogiczne do eksperymentów z wykorzystaniem zbiorów SemEval STS w pracy Ethayarajha).

---

#### 2. Kontekstowość reprezentacji słów (Ethayarajh, 2019 – *How Contextual are Contextualized Word Representations?*, D19-1006)
Analiza geometrii przestrzeni zanurzeń (*embeddings*) w kolejnych warstwach modeli ELMo, BERT i GPT-2 opiera się na czterech kluczowych obserwacjach i miarach:

* **Anizotropia (Baseline):** 
  Wektory słów w warstwach ukrytych nie są rozłożone izotropowo, lecz zajmują wąski stożek w przestrzeni wektorowej. Średnie podobieństwo kosinusowe między losowo dobranymi słowami jest istotnie większe od zera i monotonicznie rośnie w wyższych warstwach (efekt szczególnie nasilony w architekturze GPT-2). Z tego względu metryki kontekstowości wymagają korekty o wyliczoną wartość bazową anizotropii:
  $$\text{Miara}^* = \text{Miara} - \text{Baseline}$$

* **Self-Similarity ($\text{SelfSim}_l(w)$):** 
  Średnie podobieństwo kosinusowe reprezentacji tego samego słowa $w$ występującego w $n$ różnych kontekstach zdaniowych w warstwie $l$:
  $$\text{SelfSim}_l(w) = \frac{1}{n^2 - n}\sum_{j}\sum_{k \neq j} \cos(f_l(s_j, i_j), f_l(s_k, i_k))$$
  Im wyższa warstwa, tym niższa wartość skorygowanego $\text{SelfSim}^*$, co wskazuje na postępującą kontekstualizację reprezentacji. Najniższe wartości self-similarity wykazują słowa funkcyjne (*stopwords*).

* **Intra-Sentence Similarity ($\text{IntraSim}_l(s)$):** 
  Średnie podobieństwo kosinusowe wektorów słów w obrębie danego zdania $s$ względem wektora średniego tego zdania $\vec{s}_l = \frac{1}{n}\sum_i f_l(s, i)$:
  $$\text{IntraSim}_l(s) = \frac{1}{n}\sum_i \cos(\vec{s}_l, f_l(s, i))$$
  Obrazuje zachowanie reprezentacji w ramach wspólnego kontekstu: w modelu BERT reprezentacje słów w wyższych warstwach oddalają się od siebie, zachowując jednak podobieństwo powyżej losowego szumu; w GPT-2 skorygowany $\text{IntraSim}^*$ spada w pobliże zera.

* **Maximum Explainable Variance ($\text{MEV}_l(w)$):** 
  Udział wariancji macierzy wystąpień danego słowa wyjaśniany przez pierwszą składową główną (PCA / pierwszą wartość osobliwą $\sigma_1$):
  $$\text{MEV}_l(w) = \frac{\sigma_1^2}{\sum_i \sigma_i^2}$$
  Wyznacza górną granicę dokładności, z jaką pojedynczy statyczny wektor mógłby zastąpić reprezentacje kontekstowe. Po korekcie o bazę anizotropową ($\text{MEV}^*$), średnio poniżej 5% wariancji daje się wyjaśnić statycznym wektorem, co potwierdza brak redukcji reprezentacji do dyskretnych znaczeń słownikowych.

---

#### 3. Logit Lens (Nostalgebraist; Belrose et al., arXiv:2303.08112)
Technika interpretowalności mechanistycznej modeli autoregresyjnych bazująca na addytywnej naturze aktualizacji strumienia rezydualnego (*residual stream*). Przepuszczając stan ukryty $h_l$ z dowolnej warstwy pośredniej $l$ bezpośrednio przez końcową warstwę normalizacji oraz macierz rzutowania na słownik (*unembedding matrix* $W_U$), wyznacza się pośredni rozkład prawdopodobieństwa:
$$P_l(w) = \text{softmax}(W_U \cdot \text{LayerNorm}(h_l))$$
Umożliwia to bezpośrednie śledzenie formowania się ostatecznej predykcji tokenu warstwa po warstwie bez konieczności dotarcia do końca sieci.

## 1a. Anizotropia (BERT, GPT-2)

In [39]:
# Wszystkie 7 korpusów WSD
CORPORA = [
    "sherlock",
    "skladnica",
    "wikiglex",
    "emoglex",
    "walenty",
    "kpwr",
    "kpwr-100",
]

BASE_URL = (
    "https://huggingface.co/datasets/clarin-knext/wsd_polish_datasets/resolve/main/data/"
)

# Bezpośrednie pobranie plików z Hugging Face
data_files = {
    "train": [f"{BASE_URL}{corpus}_sentences.jsonl" for corpus in CORPORA]
}

# Natywny loader JSON – kompatybilny z datasets 5.0
dataset = load_dataset("json", data_files=data_files, split="train")

print("Załadowano zbiór WSD:")
print(dataset)
print("Dostępne kolumny:", dataset.column_names)

Załadowano zbiór WSD:
Dataset({
    features: ['context_file', 'sentences'],
    num_rows: 7848
})
Dostępne kolumny: ['context_file', 'sentences']


In [40]:
def extract_sentences(ds, max_sentences=1500, min_char_len=15):
    """Wyodrębnia unikalne, niepuste zdania o minimalnej zadanej długości."""
    sentences = []
    seen = set()

    for row in ds:
        sentence_data = row.get("sentences")
        if not isinstance(sentence_data, list):
            continue

        for item in sentence_data:
            if not isinstance(item, dict):
                continue

            text = item.get("text", "")
            if not isinstance(text, str):
                continue

            text = text.strip()
            if len(text) <= min_char_len or text in seen:
                continue

            seen.add(text)
            sentences.append(text)

            if len(sentences) >= max_sentences:
                return sentences

    return sentences


sentences = extract_sentences(dataset, max_sentences=1500)
print(f"Wyodrębniono {len(sentences)} unikalnych zdań do dalszej analizy.")

Wyodrębniono 1500 unikalnych zdań do dalszej analizy.


In [41]:
def extract_word_representations(model_name, sentences, max_sents=1500):
    """
    Ekstrahuje reprezentacje słów w każdej warstwie modelu (uśrednienie subtokenów).
    Zwraca:
        occurrences: lista słowników {'word', 'sent_idx', 'vecs': (n_layers, hidden_dim)}
        word_to_occ_indices: mapa słowo -> indeksy wystąpień
        sent_to_occ_indices: mapa indeks zdania -> indeksy wystąpień
    """
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = AutoModel.from_pretrained(
        model_name, output_hidden_states=True
    ).to(DEVICE)
    model.eval()

    occurrences = []
    sent_to_occ_indices = defaultdict(list)
    word_to_occ_indices = defaultdict(list)

    word_regex = re.compile(r"\b[a-ząćęłńóśźżA-ZĄĆĘŁŃÓŚŹŻ]{2,}\b")

    with torch.no_grad():
        for s_idx, sent in enumerate(sentences[:max_sents]):
            words_spans = [
                (m.group(0).lower(), m.start(), m.end())
                for m in word_regex.finditer(sent)
            ]
            if len(words_spans) < 3:
                continue

            encoded = tokenizer(
                sent,
                return_tensors="pt",
                return_offsets_mapping=True,
                truncation=True,
                max_length=256,
            )
            offsets = encoded.pop("offset_mapping")[0].cpu().numpy()
            inputs = {k: v.to(DEVICE) for k, v in encoded.items()}

            outputs = model(**inputs)
            # Kształt po złożeniu: (n_layers, seq_len, hidden_dim)
            hidden_states = torch.stack(
                [h[0].cpu() for h in outputs.hidden_states]
            ).numpy()

            for w_str, start_char, end_char in words_spans:
                # Przypisanie subtokenów do przedziałów znakowych słowa
                subtoken_indices = []
                for tok_i, (o_start, o_end) in enumerate(offsets):
                    if o_end <= o_start:
                        continue
                    # Warunek przecięcia: token nachodzi na zakres słowa [start_char, end_char]
                    if max(o_start, start_char) < min(o_end, end_char):
                        subtoken_indices.append(tok_i)

                # Reprezentacja słowa = średnia z subtokenów w danej warstwie
                word_vecs = hidden_states[:, subtoken_indices, :].mean(axis=1)

                occ_idx = len(occurrences)
                occurrences.append(
                    {"word": w_str, "sent_idx": s_idx, "vecs": word_vecs}
                )
                sent_to_occ_indices[s_idx].append(occ_idx)
                word_to_occ_indices[w_str].append(occ_idx)

    del model
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    return occurrences, word_to_occ_indices, sent_to_occ_indices


# Ekstrakcja dla HerBERT
print("Ekstrakcja reprezentacji dla modelu HerBERT...")
bert_occs, bert_w2o, bert_s2o = extract_word_representations(BERT, sentences)
print(f"Zebrano {len(bert_occs)} wystąpień słów dla HerBERT.")

# Ekstrakcja dla GPT-2
print("\nEkstrakcja reprezentacji dla modelu Polish GPT-2 Medium...")
gpt2_occs, gpt2_w2o, gpt2_s2o = extract_word_representations(GPT2, sentences)
print(f"Zebrano {len(gpt2_occs)} wystąpień słów dla GPT-2.")

Ekstrakcja reprezentacji dla modelu HerBERT...


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: allegro/herbert-base-cased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.decoder.weight             | UNEXPECTED |  | 
cls.sso.sso_relationship.bias              | UNEXPECTED |  | 
cls.predictions.decoder.bias               | UNEXPECTED |  | 
cls.sso.sso_relationship.weight            | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Zebrano 14972 wystąpień słów dla HerBERT.

Ekstrakcja reprezentacji dla modelu Polish GPT-2 Medium...


Loading weights:   0%|          | 0/292 [00:00<?, ?it/s]

[transformers] GPT2Model LOAD REPORT from: sdadas/polish-gpt2-medium
Key                                     | Status     |  | 
----------------------------------------+------------+--+-
transformer.h.{0...23}.attn.masked_bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Zebrano 14972 wystąpień słów dla GPT-2.


In [42]:
def compute_anisotropy_baseline(occurrences, target_pairs=10000, seed=42):
    """Oblicza bazową anizotropię jako średnie podobieństwo kosinusowe

    losowych par RÓŻNYCH leksykalnie słów dla każdej warstwy.
    """
    rng = np.random.default_rng(seed)
    n_layers = occurrences[0]["vecs"].shape[0]
    n_total = len(occurrences)

    # Standaryzacja wielkości liter przy porównywaniu słów
    words = np.array([occ["word"].lower() for occ in occurrences])

    valid_pairs = []
    while len(valid_pairs) < target_pairs:
        batch_size = max((target_pairs - len(valid_pairs)) * 2, 1000)
        i1 = rng.integers(0, n_total, size=batch_size)
        i2 = rng.integers(0, n_total, size=batch_size)

        mask = words[i1] != words[i2]
        valid_pairs.extend(zip(i1[mask], i2[mask]))

    idx1, idx2 = zip(*valid_pairs[:target_pairs])
    idx1, idx2 = np.array(idx1), np.array(idx2)

    # Pobranie macierzy kształtu (target_pairs, n_layers, d_model) raz
    all_vecs = np.array([occ["vecs"] for occ in occurrences])
    v1_all = all_vecs[idx1]  # shape: (target_pairs, n_layers, dim)
    v2_all = all_vecs[idx2]

    anisotropy_per_layer = []
    for l in range(n_layers):
        v1 = v1_all[:, l, :]
        v2 = v2_all[:, l, :]

        # Normalizacja L2
        v1_norm = v1 / (np.linalg.norm(v1, axis=1, keepdims=True) + 1e-12)
        v2_norm = v2 / (np.linalg.norm(v2, axis=1, keepdims=True) + 1e-12)

        cos_sims = np.sum(v1_norm * v2_norm, axis=1)
        anisotropy_per_layer.append(float(np.mean(cos_sims)))

    return np.array(anisotropy_per_layer)


bert_anisotropy = compute_anisotropy_baseline(bert_occs)
gpt2_anisotropy = compute_anisotropy_baseline(gpt2_occs)

print("Obliczono anizotropię dla obu modeli.")

Obliczono anizotropię dla obu modeli.


In [43]:
fig_aniso = make_subplots(
    rows=1,
    cols=2,
    subplot_titles=(
        "HerBERT (allegro/herbert-base-cased)",
        "Polish GPT-2 (sdadas/polish-gpt2-medium)",
    ),
)

fig_aniso.add_trace(
    go.Scatter(
        x=list(range(len(bert_anisotropy))),
        y=bert_anisotropy,
        mode="lines+markers",
        name="HerBERT",
        line=dict(color="#1f77b4"),
    ),
    row=1,
    col=1,
)

fig_aniso.add_trace(
    go.Scatter(
        x=list(range(len(gpt2_anisotropy))),
        y=gpt2_anisotropy,
        mode="lines+markers",
        name="GPT-2 Medium",
        line=dict(color="#2ca02c"),
    ),
    row=1,
    col=2,
)

fig_aniso.update_xaxes(title_text="Indeks warstwy (0 = warstwa wejściowa)")
fig_aniso.update_yaxes(
    title_text="Średnie podobieństwo kosinusowe (Anizotropia)",
    range=[0, 1.0],
)

fig_aniso.update_layout(
    title_text="1a. Anizotropia: Średnie podobieństwo kosinusowe losowych par słów",
    height=450,
    showlegend=False,
    template="plotly_white",
)

fig_aniso.show()

## 1b. Context-specificity (BERT, GPT-2)

In [46]:
# 1. Self-Similarity (Definicja 1)
def compute_self_similarity(occurrences, word_to_occ, min_occurrences=5, max_occ_per_word=30):
    n_layers = occurrences[0]["vecs"].shape[0]
    eligible_words = [w for w, idxs in word_to_occ.items() if len(idxs) >= min_occurrences]
    
    layer_self_sims = np.zeros(n_layers)
    for w in eligible_words:
        idxs = word_to_occ[w][:max_occ_per_word]
        n = len(idxs)
        for l in range(n_layers):
            vecs = np.stack([occurrences[i]["vecs"][l] for i in idxs])
            vecs_norm = vecs / (np.linalg.norm(vecs, axis=1, keepdims=True) + 1e-12)
            sim_matrix = vecs_norm @ vecs_norm.T
            # Suma poza przekątną podzielona przez (n^2 - n)
            sum_off_diag = np.sum(sim_matrix) - np.trace(sim_matrix)
            layer_self_sims[l] += sum_off_diag / (n * (n - 1))
            
    return layer_self_sims / max(len(eligible_words), 1)

In [47]:
# 2. Intra-Sentence Similarity (Definicja 2)
def compute_intra_sentence_similarity(occurrences, sent_to_occ, max_sentences=300):
    n_layers = occurrences[0]["vecs"].shape[0]
    eligible_sents = [s for s, idxs in sent_to_occ.items() if len(idxs) >= 4][:max_sentences]
    
    layer_intra_sims = np.zeros(n_layers)
    for s in eligible_sents:
        idxs = sent_to_occ[s]
        for l in range(n_layers):
            vecs = np.stack([occurrences[i]["vecs"][l] for i in idxs])
            sent_vec = np.mean(vecs, axis=0, keepdims=True)
            
            vecs_norm = vecs / (np.linalg.norm(vecs, axis=1, keepdims=True) + 1e-12)
            sent_norm = sent_vec / (np.linalg.norm(sent_vec, axis=1, keepdims=True) + 1e-12)
            
            cos_sims = np.sum(vecs_norm * sent_norm, axis=1)
            layer_intra_sims[l] += np.mean(cos_sims)
            
    return layer_intra_sims / max(len(eligible_sents), 1)

In [ ]:
# 3. Maximum Explainable Variance - MEV (Definicja 3) + baza anizotropowa dla MEV
def compute_mev_and_baseline(
    occurrences,
    word_to_occ,
    min_occurrences=5,
    max_occ_per_word=30,
    n_random_trials=100,
    seed=42,
):
  rng = np.random.default_rng(seed)
  n_layers = occurrences[0]["vecs"].shape[0]
  n_total = len(occurrences)

  eligible_words = [
      w for w, idxs in word_to_occ.items() if len(idxs) >= min_occurrences
  ]

  # 1. Obliczenie surowego MEV dla słów
  raw_mev = np.zeros(n_layers)
  sample_sizes = []

  for w in eligible_words:
    idxs = word_to_occ[w][:max_occ_per_word]
    n = len(idxs)
    sample_sizes.append(n)

    for l in range(n_layers):
      M = np.stack([occurrences[i]["vecs"][l] for i in idxs])
      _, s, _ = np.linalg.svd(M, full_matrices=False)
      raw_mev[l] += (s[0] ** 2) / (np.sum(s**2) + 1e-12)

  raw_mev /= max(len(eligible_words), 1)
  avg_n = int(np.round(np.mean(sample_sizes)))

  # 2. Losowe próbki o dokładnie takim samym rozmiarze n
  baseline_mev = np.zeros(n_layers)
  words = np.array([occ["word"].lower() for occ in occurrences])

  for _ in range(n_random_trials):
    # Losujemy avg_n różnych słów
    chosen_idxs = []
    while len(chosen_idxs) < avg_n:
      candidate = rng.integers(0, n_total)
      if candidate not in chosen_idxs:
        chosen_idxs.append(candidate)

    for l in range(n_layers):
      M_rand = np.stack([occurrences[i]["vecs"][l] for i in chosen_idxs])
      _, s_rand, _ = np.linalg.svd(M_rand, full_matrices=False)
      baseline_mev[l] += (s_rand[0] ** 2) / (np.sum(s_rand**2) + 1e-12)

  baseline_mev /= n_random_trials

  # MEV* = surowe MEV - anizotropowy baseline MEV
  adj_mev = np.maximum(0, raw_mev - baseline_mev)

  return raw_mev, adj_mev

In [52]:
# Obliczenia dla HerBERT
bert_self_raw = compute_self_similarity(bert_occs, bert_w2o)
bert_self_adj = bert_self_raw - bert_anisotropy

bert_intra_raw = compute_intra_sentence_similarity(bert_occs, bert_s2o)
bert_intra_adj = bert_intra_raw - bert_anisotropy

bert_mev_raw, bert_mev_adj = compute_mev_and_baseline(bert_occs, bert_w2o)

# Obliczenia dla GPT-2
gpt2_self_raw = compute_self_similarity(gpt2_occs, gpt2_w2o)
gpt2_self_adj = gpt2_self_raw - gpt2_anisotropy

gpt2_intra_raw = compute_intra_sentence_similarity(gpt2_occs, gpt2_s2o)
gpt2_intra_adj = gpt2_intra_raw - gpt2_anisotropy

gpt2_mev_raw, gpt2_mev_adj = compute_mev_and_baseline(gpt2_occs, gpt2_w2o)

In [54]:
# Wizualizacja 1b: Self-Similarity, Intra-Sentence Similarity oraz MEV (skorygowane o anizotropię)
fig_ctx = make_subplots(
    rows=3, cols=2,
    subplot_titles=(
        "HerBERT: Self-Similarity (skorygowane)", "GPT-2: Self-Similarity (skorygowane)",
        "HerBERT: Intra-Sentence Similarity (skorygowane)", "GPT-2: Intra-Sentence Similarity (skorygowane)",
        "HerBERT: MEV (skorygowane i surowe)", "GPT-2: MEV (skorygowane i surowe)"
    ),
    vertical_spacing=0.12
)

# Wiersz 1: Self-Similarity
fig_ctx.add_trace(go.Scatter(x=list(range(len(bert_self_adj))), y=bert_self_adj, mode="lines+markers", name="HerBERT SelfSim*"), row=1, col=1)
fig_ctx.add_trace(go.Scatter(x=list(range(len(gpt2_self_adj))), y=gpt2_self_adj, mode="lines+markers", name="GPT-2 SelfSim*"), row=1, col=2)

# Wiersz 2: Intra-Sentence Similarity
fig_ctx.add_trace(go.Scatter(x=list(range(len(bert_intra_adj))), y=bert_intra_adj, mode="lines+markers", name="HerBERT IntraSim*"), row=2, col=1)
fig_ctx.add_trace(go.Scatter(x=list(range(len(gpt2_intra_adj))), y=gpt2_intra_adj, mode="lines+markers", name="GPT-2 IntraSim*"), row=2, col=2)

# Wiersz 3: MEV
fig_ctx.add_trace(go.Bar(x=list(range(len(bert_mev_adj))), y=bert_mev_adj, name="HerBERT MEV* (skorygowane)"), row=3, col=1)
fig_ctx.add_trace(go.Scatter(x=list(range(len(bert_mev_raw))), y=bert_mev_raw, mode="lines+markers", name="HerBERT MEV (surowe)"), row=3, col=1)

fig_ctx.add_trace(go.Bar(x=list(range(len(gpt2_mev_adj))), y=gpt2_mev_adj, name="GPT-2 MEV* (skorygowane)"), row=3, col=2)
fig_ctx.add_trace(go.Scatter(x=list(range(len(gpt2_mev_raw))), y=gpt2_mev_raw, mode="lines+markers", name="GPT-2 MEV (surowe)"), row=3, col=2)

fig_ctx.update_xaxes(title_text="Warstwa")
fig_ctx.update_layout(title_text="1b. Zależność od kontekstu (Context-Specificity) per warstwa", height=950)
fig_ctx.show()

## 2. Logit Lens (`sdadas/polish-gpt2-medium`)

In [57]:
# 2a-b. Zestaw 24 promptów: 12 faktograficznych ('fakt') oraz 12 gramatycznych ('gramatyka')
PROMPTS = [
    # --- Fakty (12 promptów) ---
    {"prompt": "Stolicą Polski jest", "typ": "fakt", "expected": " Warszawa"},
    {"prompt": "Stolicą Francji jest", "typ": "fakt", "expected": " Paryż"},
    {"prompt": "Stolicą Niemiec jest", "typ": "fakt", "expected": " Berlin"},
    {"prompt": "Stolicą Włoch jest", "typ": "fakt", "expected": " Rzym"},
    {"prompt": "Stolicą Hiszpanii jest", "typ": "fakt", "expected": " Madryt"},
    {"prompt": "Najdłuższą rzeką w Polsce jest", "typ": "fakt", "expected": " Wisła"},
    {"prompt": "Autorem epopei Pan Tadeusz jest Adam", "typ": "fakt", "expected": " Mickiewicz"},
    {"prompt": "Walutą obowiązującą w Polsce jest polski", "typ": "fakt", "expected": " złoty"},
    {"prompt": "Najwyższym szczytem w polskich Tatrach są", "typ": "fakt", "expected": " Rysy"},
    {"prompt": "Ziemia krąży wokół", "typ": "fakt", "expected": " Słońca"},
    {"prompt": "Pierwszym miastem stołecznym Polski było", "typ": "fakt", "expected": " Gniezno"},
    {"prompt": "Woda zamarza w temperaturze zero stopni", "typ": "fakt", "expected": " Celsjusza"},
    
    # --- Gramatyka / zgodność fleksyjna i składniowa (12 promptów) ---
    {"prompt": "Wczoraj poszedłem do sklepu, a moja siostra też tam", "typ": "gramatyka", "expected": " poszła"},
    {"prompt": "Te małe dzieci bardzo głośno", "typ": "gramatyka", "expected": " płaczą"},
    {"prompt": "Nie mam dzisiaj ani trochę", "typ": "gramatyka", "expected": " czasu"},
    {"prompt": "Przyglądam się temu wysokiemu", "typ": "gramatyka", "expected": " budynkowi"},
    {"prompt": "Gdybym miał więcej wolnego czasu, to", "typ": "gramatyka", "expected": " bym"},
    {"prompt": "Książka, którą wczoraj wieczorem", "typ": "gramatyka", "expected": " czytałem"},
    {"prompt": "Z powodu silnego deszczu musieliśmy zostać w", "typ": "gramatyka", "expected": " domu"},
    {"prompt": "W pokoju stoją dwa drewniane", "typ": "gramatyka", "expected": " stoły"},
    {"prompt": "W pokoju stoi pięć drewnianych", "typ": "gramatyka", "expected": " stołów"},
    {"prompt": "Ona codziennie rano pije gorącą", "typ": "gramatyka", "expected": " kawę"},
    {"prompt": "Rozmawialiśmy o naszych wspólnych", "typ": "gramatyka", "expected": " planach"},
    {"prompt": "Ten nowy student okazał się bardzo", "typ": "gramatyka", "expected": " zdolny"},
]

tokenizer_gpt2 = AutoTokenizer.from_pretrained(GPT2)
model_gpt2 = AutoModelForCausalLM.from_pretrained(GPT2, output_hidden_states=True).to(DEVICE)
model_gpt2.eval()

Loading weights:   0%|          | 0/292 [00:00<?, ?it/s]

[transformers] GPT2LMHeadModel LOAD REPORT from: sdadas/polish-gpt2-medium
Key                                     | Status     |  | 
----------------------------------------+------------+--+-
transformer.h.{0...23}.attn.masked_bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


GPT2LMHeadModel(
  (transformer): GPT2Model(
    (wte): Embedding(51200, 1024)
    (wpe): Embedding(2048, 1024)
    (drop): Dropout(p=0.1, inplace=False)
    (h): ModuleList(
      (0-23): 24 x GPT2Block(
        (ln_1): LayerNorm((1024,), eps=1e-05, elementwise_affine=True, bias=True)
        (attn): GPT2Attention(
          (c_attn): Conv1D(nf=3072, nx=1024)
          (c_proj): Conv1D(nf=1024, nx=1024)
          (attn_dropout): Dropout(p=0.1, inplace=False)
          (resid_dropout): Dropout(p=0.1, inplace=False)
        )
        (ln_2): LayerNorm((1024,), eps=1e-05, elementwise_affine=True, bias=True)
        (mlp): GPT2MLP(
          (c_fc): Conv1D(nf=4096, nx=1024)
          (c_proj): Conv1D(nf=1024, nx=4096)
          (act): FastGELUActivation()
          (dropout): Dropout(p=0.1, inplace=False)
        )
      )
    )
    (ln_f): LayerNorm((1024,), eps=1e-05, elementwise_affine=True, bias=True)
  )
  (lm_head): Linear(in_features=1024, out_features=51200, bias=False)
)

In [58]:
# 2c. Funkcja wykonująca projekcję Logit Lens dla pojedynczego promptu:
# stan ukryty ostatniego tokenu -> ln_f -> lm_head -> rozkład po słowniku (softmax)
def run_logit_lens(prompt_text, model, tokenizer):
    inputs = tokenizer(prompt_text, return_tensors="pt").to(DEVICE)
    with torch.no_grad():
        outputs = model(**inputs)
        # outputs.hidden_states: 25 tensorów dla gpt2-medium (warstwa 0 + 24 bloki Transformera)
        hidden_states = outputs.hidden_states
        
        layer_probs = []
        layer_top1_tokens = []
        layer_top1_probs = []
        
        # Token przewidziany na samym wyjściu modelu (ostatnia warstwa)
        final_logits = outputs.logits[0, -1, :]
        final_pred_id = int(torch.argmax(final_logits).item())
        final_pred_token = tokenizer.decode([final_pred_id])
        
        ranks_of_final = []
        probs_of_final = []
        
        for l_idx, h in enumerate(hidden_states):
            last_token_h = h[0, -1, :]  # (hidden_dim,)
            
            # W ostatniej warstwie huggingface GPT2Model aplikuje już ln_f do hidden_states[-1],
            # natomiast warstwy pośrednie (0..L-1) wymagają jawnego przepuszczenia przez ln_f
            if l_idx < len(hidden_states) - 1:
                normed_h = model.transformer.ln_f(last_token_h)
            else:
                normed_h = last_token_h
                
            logits = model.lm_head(normed_h)  # (vocab_size,)
            probs = torch.softmax(logits, dim=-1)
            
            top_prob, top_id = torch.max(probs, dim=-1)
            layer_top1_tokens.append(tokenizer.decode([int(top_id.item())]).strip() or "<space>")
            layer_top1_probs.append(float(top_prob.item()))
            
            # Ranga tokenu przewidywanego na wyjściu modelu (1 = najwyższe prawdopodobieństwo)
            target_logit = logits[final_pred_id]
            rank = int(torch.sum(logits > target_logit).item()) + 1
            
            ranks_of_final.append(rank)
            probs_of_final.append(float(probs[final_pred_id].item()))
            
    return {
        "final_token": final_pred_token,
        "top1_tokens": layer_top1_tokens,
        "top1_probs": layer_top1_probs,
        "ranks_of_final": ranks_of_final,
        "probs_of_final": probs_of_final
    }

results = []
for item in PROMPTS:
    res = run_logit_lens(item["prompt"], model_gpt2, tokenizer_gpt2)
    res.update(item)
    results.append(res)

In [62]:
# 2d. Heatmapa najbardziej prawdopodobnych tokenów w kolejnych warstwach dla wybranych promptów
selected_indices = [0, 1, 5, 6, 12, 14, 15, 20]  # 4 faktograficzne i 4 gramatyczne
selected_res = [results[i] for i in selected_indices]

z_probs = np.array([r["top1_probs"] for r in selected_res])
raw_tokens = np.array([r["top1_tokens"] for r in selected_res])

# Max 7 znaków, by nie nachodziły na sąsiednie komórki
display_tokens = np.vectorize(lambda t: t[:6] + "…" if len(t) > 7 else t)(raw_tokens)

# Sformatowane etykiety osi Y
y_labels = [f"[{r['typ'].upper()}] {r['prompt']} ➔ <b>{r['final_token'].strip()}</b>" for r in selected_res]
x_layers = [f"L{i}" for i in range(z_probs.shape[1])]

# Pełny tekst widoczny po najechaniu kursorem
hover_text = [[
    f"Warstwa: {x_layers[col]}<br>Prompt: {selected_res[row]['prompt']}<br>Token: <b>'{raw_tokens[row, col]}'</b><br>Prawdopodobieństwo: {z_probs[row, col]:.3f}"
    for col in range(z_probs.shape[1])
] for row in range(len(selected_res))]

fig_heatmap = go.Figure(data=go.Heatmap(
    z=z_probs,
    x=x_layers,
    y=y_labels,
    text=display_tokens,
    texttemplate="%{text}",
    textfont={"size": 8.5, "family": "Arial"},
    hoverinfo="text",
    hovertext=hover_text,
    colorscale="Viridis",
    colorbar=dict(title="P(top-1)")
))

fig_heatmap.update_layout(
    title="2d. Logit Lens: Najbardziej prawdopodobny token w kolejnych warstwach",
    xaxis=dict(
        title="Warstwa modelu (sdadas/polish-gpt2-medium)",
        tickmode="linear",
        tickangle=0,
        side="bottom"
    ),
    yaxis=dict(
        autorange="reversed",  # Od góry do dołu (zgodnie z listą promptów)
        tickfont={"size": 11}
    ),
    width=1500,   # Duża szerokość rozciąga 25 kolumn w poziomie
    height=600,   # Odpowiednia wysokość na 8 wierszy
    margin=dict(l=380, r=40, t=60, b=60)  # Margines z lewej na długie teksty promptów
)

fig_heatmap.show()

In [63]:
# 2e. Ranga i prawdopodobieństwo tokenu przewidywanego na wyjściu modelu, per warstwa, uśrednione po promptach (osobno fakt vs gramatyka)
n_layers_gpt2 = len(results[0]["ranks_of_final"])
layers_idx = list(range(n_layers_gpt2))

fakt_ranks = np.mean([r["ranks_of_final"] for r in results if r["typ"] == "fakt"], axis=0)
gram_ranks = np.mean([r["ranks_of_final"] for r in results if r["typ"] == "gramatyka"], axis=0)

fakt_probs = np.mean([r["probs_of_final"] for r in results if r["typ"] == "fakt"], axis=0)
gram_probs = np.mean([r["probs_of_final"] for r in results if r["typ"] == "gramatyka"], axis=0)

fig_metrics = make_subplots(
    rows=1, cols=2,
    subplot_titles=(
        "Średnia ranga docelowego tokenu (skala log)",
        "Średnie prawdopodobieństwo docelowego tokenu"
    )
)

fig_metrics.add_trace(go.Scatter(x=layers_idx, y=fakt_ranks, mode="lines+markers", name="Fakty (ranga)"), row=1, col=1)
fig_metrics.add_trace(go.Scatter(x=layers_idx, y=gram_ranks, mode="lines+markers", name="Gramatyka (ranga)"), row=1, col=1)

fig_metrics.add_trace(go.Scatter(x=layers_idx, y=fakt_probs, mode="lines+markers", name="Fakty (prawdopodobieństwo)"), row=1, col=2)
fig_metrics.add_trace(go.Scatter(x=layers_idx, y=gram_probs, mode="lines+markers", name="Gramatyka (prawdopodobieństwo)"), row=1, col=2)

fig_metrics.update_xaxes(title_text="Indeks warstwy")
fig_metrics.update_yaxes(title_text="Średnia ranga (mniej = lepiej)", type="log", row=1, col=1)
fig_metrics.update_yaxes(title_text="Średnie prawdopodobieństwo P(y_final)", range=[0, 1.0], row=1, col=2)
fig_metrics.update_layout(
    title_text="2e. Ewolucja rangi i prawdopodobieństwa tokenu wyjściowego: Zdania faktograficzne vs Gramatyczne",
    height=450
)
fig_metrics.show()

### Wnioski 

#### 1. Geometria i kontekstowość reprezentacji (HerBERT vs Polish GPT-2 Medium)
* **Anizotropia:** Zgodnie z obserwacjami Ethayarajha (2019), w obu polskich modelach wektory słów w warstwach ukrytych wykazują silną anizotropię (reprezentacje zajmują wąski stożek w przestrzeni wektorowej, a średnie podobieństwo losowych słów przekracza 0.5–0.7). W modelu `allegro/herbert-base-cased` anizotropia rośnie gwałtownie w warstwach 1–3 i utrzymuje wysoki poziom aż do warstwy 12. W `sdadas/polish-gpt2-medium` anizotropia utrzymuje się na poziomie ok. 0.5 w warstwach środkowych, rośnie pod koniec sieci (warstwy 18–23), po czym gwałtownie załamuje się w warstwie wyjściowej (L24) do poziomu ok. 0.22 tuż przed projekcją na przestrzeń słownika.
* **Self-Similarity ($\text{SelfSim}^*$):** Po odjęciu bazy anizotropowej podobieństwo tego samego leksykalnie słowa w odmiennych kontekstach systematycznie maleje wraz z głębokością sieci w obu architekturach. Wyższe warstwy tworzą wektory silniej uzależnione od konkretnego zdania i jego semantyki.
* **Intra-Sentence Similarity ($\text{IntraSim}^*$):** W modelu HerBERT słowa należące do tego samego zdania zachowują wyższe podobieństwo niż losowe pary słów ($\text{IntraSim}^* > 0$), choć w wyższych warstwach ich wektory stopniowo różnicują swoje role składniowe. W modelu GPT-2 skorygowane podobieństwo wewnątrzzdaniowe w warstwach środkowych spada w pobliże zera, co dowodzi, że transformer autoregresyjny nie skupia reprezentacji wokół jednego średniego wektora zdania.
* **Maximum Explainable Variance ($\text{MEV}^*$):** Po odjęciu adekwatnego wymiarowo baseline'u anizotropowego, pierwsza składowa główna wyjaśnia jedynie ułamek (w wyższych warstwach rzędu 5–10%) wariancji wystąpień danego słowa. Potwierdza to, że reprezentacje kontekstowe w modelach języka są ciągłe i niesprowadzalne do statycznego wektora czy dyskretnego zestawu znaczeń słownikowych.

---

#### 2. Logit Lens dla sdadas/polish-gpt2-medium (Fakty vs Gramatyka)
* **Wczesne warstwy (L0–L8) – echa kontekstu i kopiowanie:** Rozkład prawdopodobieństwa jest zdominowany przez powtórzenia ostatniego tokenu wejściowego (np. `jest`, `tam`, `Adam`) lub częste tokeny funkcyjne. Średnia ranga właściwego tokenu docelowego w słowniku wynosi od kilkuset do ponad 10 000 pozycji.
* **Warstwy środkowe (L9–L16) – prymat ograniczeń gramatycznych:** Ograniczenia składniowe i fleksyjne (wymagany przypadek po przyimku/liczebniku, zgodność rodzaju i liczby) są rozwiązywane znacznie wcześniej niż wiedza faktograficzna. Ranga poprawnej formy gramatycznej spada do czołowych pozycji już w okolicach warstw 12–15, podczas gdy dla faktów proces ten jest przesunięty w głąb sieci.
* **Warstwy górne (L17–L24) – krystalizacja wiedzy faktograficznej:** Wydobycie konkretnej wiedzy encyklopedycznej (stolica państwa, nazwisko autora, nazwa rzeki) wymaga głębszej ekstrakcji relacji semantycznych przez wyższe warstwy uwagi i feed-forward. Spadek rangi do top-1 dla faktów następuje skokowo w warstwach 17–22, osiągając ostatecznie równie wysoką pewność predykcji co w zadaniach gramatycznych.